In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

warnings.filterwarnings('ignore')

patient_vals=pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
params=pd.read_csv("IntermediateResults/venex_weighted_params.csv")
merged=pd.merge(patient_vals, params, left_on="Study_subject", right_on="ID")
# =====================================================================
# 1. SETUP
# =====================================================================
sns.set_theme(style="white")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

fig, axes = plt.subplots(3, 3, figsize=(14, 13))

# Okabe-Ito colorblind-accessible palettes
group_colors = {
    "Low Risk Profile": "#56B4E9",   # Sky Blue
    "High Risk Profile": "#D55E00",  # Vermilion
}

risk_palette = {
    'Favorable': '#0072B2',         'Low Risk': '#0072B2',
    'Intermediate': '#E69F00',      'Intermediate Risk': '#E69F00',
    'Adverse': '#D55E00',           'High Risk': '#D55E00'
}

group_linestyles = {
    "Low Risk Profile": "-",
    "High Risk Profile": "--",
}

risk_linestyles = {
    'Favorable': '-',                'Low Risk': '-',
    'Intermediate': '-.',            'Intermediate Risk': '-.',
    'Adverse': '--',                 'High Risk': '--'
}

# =====================================================================
# 2. DATA PIPELINE 
# =====================================================================
df_processed = merged.copy()

for col in ["pHSC_weighted", "coef_weighted", "bm_blasts", "b_neut"]:
    z_col = col.replace("_weighted", "") + "_z" if "weighted" in col else col + "_z"
    df_processed[z_col] = (df_processed[col] - df_processed[col].mean()) / df_processed[col].std()

df_processed["OS_time"] = pd.to_numeric(df_processed["OS_time"], errors="coerce")
df_processed["OS_status"] = pd.to_numeric(df_processed["OS_status"], errors="coerce").fillna(0).astype(int)

if "Arm" not in df_processed.columns and "disease_stage" in df_processed.columns:
    df_processed["Arm"] = np.where(
        df_processed["disease_stage"].astype(str).str.strip() == "De novo / ND AML",
        "Arm1",
        "Arm2"
    )

master_cols = [
    "OS_time", "OS_status", "Arm", "bm_blasts_z", "b_neut_z", "pHSC_z", "coef_z",
    "TP53", "FLT3_itd", "NRAS", "KRAS", "PTPN11", "NPM1", "IDH1", "IDH2", "DDX41"
]
available_cols = [c for c in master_cols if c in df_processed.columns]
df_master_clean = df_processed[available_cols].dropna(
    subset=["OS_time", "OS_status", "pHSC_z", "coef_z", "bm_blasts_z", "b_neut_z"]
).copy().reset_index(drop=True)

# =====================================================================
# 3. LOG-RANK
# =====================================================================
def find_optimal_cutpoint(df, score_col, time_col, event_col):
    scores = np.sort(df[score_col].unique())
    if len(scores) < 2: return scores[0], 1.0

    low_bound, high_bound = np.percentile(scores, 15), np.percentile(scores, 85)
    candidate_scores = [s for s in scores if low_bound <= s <= high_bound] or list(scores)

    best_stat, best_cut, best_p = -1.0, candidate_scores[0], 1.0

    for cut in candidate_scores:
        group = np.where(df[score_col] <= cut, 0, 1)
        if len(np.unique(group)) < 2: continue

        t0 = np.asarray(df.loc[group == 0, time_col], dtype=float).ravel()
        t1 = np.asarray(df.loc[group == 1, time_col], dtype=float).ravel()
        e0 = np.asarray(df.loc[group == 0, event_col], dtype=int).ravel()
        e1 = np.asarray(df.loc[group == 1, event_col], dtype=int).ravel()

        try:
            res = logrank_test(t0, t1, e0, e1)
            if float(res.test_statistic) > best_stat:
                best_stat, best_p, best_cut = float(res.test_statistic), float(res.p_value), cut
        except: continue

    return best_cut, best_p

# =====================================================================
# 4. PIPELINE 
# =====================================================================
def process_and_plot_cohort(cohort_df, cohort_label, axes_row, is_mechanistic, col_idx, label_letter):
    ax = axes_row[col_idx]

    # Panel Letters
    ax.text(-0.08, 1.06, label_letter, transform=ax.transAxes, fontsize=18, fontweight="bold", va="top", ha="right")

    try:
        cph = CoxPHFitter(penalizer=0.01)
        formula = "pHSC_z  + coef_z" if is_mechanistic else "bm_blasts_z + b_neut_z"
        cph.fit(cohort_df, duration_col="OS_time", event_col="OS_status", formula=formula)
        score_col = "score_mech" if is_mechanistic else "score_raw"

        cohort_df[score_col] = np.asarray(cph.predict_log_partial_hazard(cohort_df)).ravel()
        cut_point, lr_p_val = find_optimal_cutpoint(cohort_df, score_col, "OS_time", "OS_status")

        cohort_df["final_group"] = np.where(cohort_df[score_col] <= cut_point, "Low Risk Profile", "High Risk Profile")

        for grp in ["Low Risk Profile", "High Risk Profile"]:
            mask = cohort_df["final_group"] == grp
            n_grp = mask.sum()
            if n_grp > 0:
                kmf = KaplanMeierFitter()
                kmf.fit(durations=cohort_df.loc[mask, "OS_time"], event_observed=cohort_df.loc[mask, "OS_status"], label=f"{grp.split()[0]} Risk (N={n_grp})")
                kmf.plot_survival_function(
                    ax=ax, color=group_colors[grp], linestyle=group_linestyles[grp],
                    linewidth=2.5, ci_show=False, show_censors=True, censor_styles={'ms': 6, 'marker': '+' if grp == "Low Risk Profile" else 'x'}
                )

        ax.set_xlabel("Time Since Treatment (Days)", fontsize=11, fontweight="bold", labelpad=8)
        ax.set_xlim(0, float(cohort_df["OS_time"].max()))
        ax.set_ylim(-0.02, 1.02)
        ax.tick_params(axis="both", labelsize=10)
        ax.grid(True, alpha=0.15, linestyle="--")

        c_idx = float(cph.concordance_index_)
        split_p_string = f"$p = {lr_p_val:.4e}$" if lr_p_val < 0.001 else f"$p = {lr_p_val:.4f}$"
        metric_box_text = f"Cohort: {cohort_label}\nC-Index = {c_idx:.3f}\nLog-Rank {split_p_string}"

        ax.text(
            0.45, 0.68, metric_box_text, transform=ax.transAxes, fontsize=9.5, fontweight="bold", linespacing=1.4,
            bbox=dict(facecolor="white", alpha=0.95, edgecolor="#cccccc", boxstyle="round,pad=0.4")
        )
        ax.legend(loc="upper right", frameon=True, fontsize=9.5, facecolor="white", edgecolor="none")

    except Exception as e:
        ax.text(0.5, 0.5, f"Error:\n{str(e)}", ha="center", va="center", color="darkred", transform=ax.transAxes, fontsize=10)

    return cohort_df

# =====================================================================
# 5. EXECUTION: ROWS 1 & 2
# =====================================================================
cohort_configs = [
    ("All samples", df_master_clean.copy()),
    ("Arm 1 (De novo)", df_master_clean[df_master_clean["Arm"] == "Arm1"].copy()),
    ("Arm 2 (sAML / R/R)", df_master_clean[df_master_clean["Arm"] == "Arm2"].copy()),
]

df_denovo_mech = None

for col_idx, (name, df_subset) in enumerate(cohort_configs):
    process_and_plot_cohort(df_subset.reset_index(drop=True), name, axes[0], False, col_idx, ["A", "B", "C"][col_idx])
    df_mech = process_and_plot_cohort(df_subset.reset_index(drop=True), name, axes[1], True, col_idx, ["D", "E", "F"][col_idx])
    if name == "Arm 1 (De novo)": df_denovo_mech = df_mech

# =====================================================================
# 6. EXECUTION: ROW 3 (ELN COMPARISONS)
# =====================================================================
if df_denovo_mech is not None:
    df_denovo = df_denovo_mech.copy()

    cut_33 = np.percentile(df_denovo['score_mech'], 33.333)
    cut_66 = np.percentile(df_denovo['score_mech'], 66.667)
    df_denovo['AML_DT_Risk'] = np.where(df_denovo['score_mech'] <= cut_33, 'Low Risk', np.where(df_denovo['score_mech'] <= cut_66, 'Intermediate Risk', 'High Risk'))

    def classify_eln(r):
        if r.get('TP53', 0) == 1: return 'Adverse'
        if r.get('FLT3_itd', 0) == 1 or r.get('NRAS', 0) == 1 or r.get('KRAS', 0) == 1: return 'Intermediate'
        return 'Favorable'

    def classify_refined(r):
        if r.get('TP53', 0) == 1 or r.get('KRAS', 0) == 1 or r.get('PTPN11', 0) == 1: return 'Adverse'
        if r.get('FLT3_itd', 0) == 1 or r.get('NRAS', 0) == 1: return 'Intermediate'
        if r.get('NPM1', 0) == 1 or r.get('IDH1', 0) == 1 or r.get('IDH2', 0) == 1 or r.get('DDX41', 0) == 1: return 'Favorable'
        return 'Intermediate'

    df_denovo['ELN_2024'] = df_denovo.apply(classify_eln, axis=1)
    df_denovo['Refined_ELN_2024'] = df_denovo.apply(classify_refined, axis=1)

    configs = [
        {'col': 'ELN_2024', 'order': ['Favorable', 'Intermediate', 'Adverse']},
        {'col': 'Refined_ELN_2024', 'order': ['Favorable', 'Intermediate', 'Adverse']},
        {'col': 'AML_DT_Risk', 'order': ['Low Risk', 'Intermediate Risk', 'High Risk']}
    ]

    for col_idx, config in enumerate(configs):
        ax = axes[2, col_idx]
        for tier in config['order']:
            df_tier = df_denovo[df_denovo[config['col']] == tier]
            if len(df_tier) > 0:
                kmf = KaplanMeierFitter()
                kmf.fit(durations=df_tier['OS_time'], event_observed=df_tier['OS_status'], label=f"{tier} (N={len(df_tier)})")
                kmf.plot_survival_function(ax=ax, color=risk_palette[tier], linestyle=risk_linestyles[tier], linewidth=2.5, ci_show=False, show_censors=True, censor_styles={'ms': 6, 'marker': '+'})

        codes = df_denovo[config['col']].map({t: i for i, t in enumerate(config['order'])})
        try:
            p_val = multivariate_logrank_test(df_denovo['OS_time'], codes, df_denovo['OS_status']).p_value
            c_idx = concordance_index(df_denovo['OS_time'], -codes, df_denovo['OS_status'])
            badge = f"Log-Rank $p = {p_val:.4e}$" if p_val < 0.001 else f"Log-Rank $p = {p_val:.3f}$"
            badge += f"\nC-index = {c_idx:.3f}"
        except: badge = "N/A"

        ax.set_xlabel('Time Since Treatment (Days)', fontsize=11, fontweight='bold', labelpad=8)
        ax.set_xlim(0, 1150)
        ax.set_ylim(-0.02, 1.02)
        ax.tick_params(axis='both', labelsize=10)
        ax.grid(True, alpha=0.15, linestyle="--")

        ax.text(0.48, 0.12, badge, transform=ax.transAxes, fontsize=9.5, fontweight='bold', linespacing=1.4, bbox=dict(facecolor='white', alpha=0.95, edgecolor='#cccccc', boxstyle='round,pad=0.4'))
        ax.legend(loc='upper right', frameon=True, fontsize=9.5, facecolor='white', edgecolor='none')
        ax.text(-0.08, 1.06, ["G", "H", "I"][col_idx], transform=ax.transAxes, fontsize=18, fontweight='bold', va='top', ha='right')

# =====================================================================
# 7.  SAVE
# =====================================================================
axes[0, 0].set_ylabel("Overall Survival Probability\n[Raw Lab Measurements Model]", fontsize=12, fontweight="bold", labelpad=12)
axes[1, 0].set_ylabel("Overall Survival Probability\n[Mechanistic ODE Kinetics Model]", fontsize=12, fontweight="bold", labelpad=12)
axes[2, 0].set_ylabel("Overall Survival Probability\n[Clinical Risk Frameworks]", fontsize=12, fontweight="bold", labelpad=12)

sns.despine(trim=True)
plt.tight_layout(h_pad=2.0, w_pad=1.5)

os.makedirs('Figures', exist_ok=True)
plt.savefig('Figures/Figure2.png', dpi=300, bbox_inches='tight')
plt.savefig('Figures/Figure2.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()